# 🌸 Phân Loại Ảnh Hoa (Hoa Hồng, Hoa Cúc, Hoa Ly) với AlexNet — PyTorch

Notebook này **chỉ đóng vai trò trình diễn**: toàn bộ logic nằm trong package `src/`
(tải dữ liệu, transforms, kiến trúc model, huấn luyện, đánh giá, inference).

> **Lưu ý:** Notebook không copy-paste lại code. Muốn sửa hành vi, sửa trong `src/`
> rồi chạy lại notebook — script và notebook luôn dùng chung một nguồn sự thật.

**Cấu trúc liên quan**
- `src/data/downloader.py` — tải & lọc dữ liệu Kaggle
- `src/data/transforms.py` — augmentation / preprocessing
- `src/models/alexnet.py` — AlexNet from scratch & pre-trained
- `src/pipelines/trainer.py` — vòng lặp huấn luyện + early stopping
- `src/pipelines/evaluator.py` — đánh giá & confusion matrix
- `src/pipelines/predictor.py` — engine inference

## 0. Thiết lập môi trường

Chạy notebook từ thư mục gốc của dự án (hoặc từ `notebooks/`).
Trên Google Colab, clone repo về trước rồi mới chạy các cell bên dưới.

In [ ]:
# Trên Colab: bỏ comment 2 dòng dưới và thay URL repo của bạn
# !git clone https://github.com/<user>/<repo>.git
# %cd <repo>

import os
import sys

# Tự tìm thư mục gốc dự án (chứa package src/)
PROJECT_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)

import torch
import config
from src.utils.console import setup_console
from src.utils.logger import setup_logging
from src.utils.seed import set_seed

setup_console()
setup_logging()
set_seed(config.SEED)

print('torch:', torch.__version__)
print('device:', config.DEVICE)
print('classes:', config.TARGET_CLASSES)
print('use_pretrained:', config.USE_PRETRAINED)

## 1. Tải & chuẩn bị dữ liệu

`prepare_flower_dataset()` sẽ:
1. Dùng dữ liệu có sẵn trong `flower_data_3classes/` nếu đã có;
2. nếu chưa, tải bộ Kaggle (cần `kagglehub`);
3. nếu cả hai đều không được, **raise lỗi rõ ràng** (không còn tự sinh ảnh random).

In [ ]:
from src.data.dataset import prepare_flower_dataset

data_ready = prepare_flower_dataset()
print('Dataset sẵn sàng:', data_ready)

## 2. DataLoader & Data Augmentation

In [ ]:
from src.data.transforms import get_transforms

train_tf, val_tf = get_transforms()
print('Train transforms:')
print(train_tf)
print()
print('Val/Test transforms:')
print(val_tf)

In [ ]:
from src.data.dataloader import create_dataloaders

train_loader, val_loader, test_loader, class_names = create_dataloaders()
print('classes:', class_names)
print('train/val/test:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

images, labels = next(iter(train_loader))
print('1 batch:', images.shape, labels.shape)

## 3. Kiến trúc mô hình AlexNet

- `USE_PRETRAINED = False` → `AlexNetFromScratch`
- `USE_PRETRAINED = True` → AlexNet pre-trained ImageNet, thay lớp cuối

> Kết quả tham chiếu đo được: scratch ≈ 80–85% accuracy, pre-trained ≈ 94% trên tập test.

In [ ]:
from src.models.builder import build_model

model = build_model(
    use_pretrained=config.USE_PRETRAINED,
    num_classes=len(class_names),
    device=config.DEVICE,
)
print(model)

## 4. Huấn luyện

Dùng trực tiếp `train_model()` của dự án: Adam + ReduceLROnPlateau + early stopping,
tự động lưu checkpoint tốt nhất vào `checkpoints/`.

In [ ]:
import torch.nn as nn
import torch.optim as optim

from src.pipelines.trainer import save_training_artifacts, train_model

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=config.NUM_EPOCHS,
    device=config.DEVICE,
)

save_training_artifacts(history, class_names)

## 5. Trực quan hóa quá trình huấn luyện

In [ ]:
from IPython.display import Image as IPyImage, display

# save_training_artifacts() o cell tren da ve san bieu do nay
curves_path = os.path.join(config.OUTPUT_DIR, 'training_curves.png')
display(IPyImage(filename=curves_path))


## 6. Đánh giá trên tập test & Confusion Matrix

`evaluate_model()` load checkpoint trong `checkpoints/`, in
`classification_report` và lưu `outputs/confusion_matrix.png`.

In [ ]:
from src.pipelines.evaluator import evaluate_model

metrics = evaluate_model()
metrics

In [ ]:
cm_path = os.path.join(config.OUTPUT_DIR, 'confusion_matrix.png')
if not os.path.exists(cm_path):
    print('Chưa có confusion matrix, hãy chạy cell trên.')
else:
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=cm_path))
    except ImportError:
        print('Không có IPython để hiển thị inline, xem ảnh tại:', cm_path)

## 7. Thử nghiệm dự đoán ảnh bất kỳ

`FlowerPredictor` **tự đọc kiến trúc và thứ tự lớp từ checkpoint**, nên load được
cả bản scratch lẫn bản pre-trained.

In [ ]:
from src.pipelines.predictor import FlowerPredictor

predictor = FlowerPredictor()
print('is_trained:', predictor.is_trained)
print('classes (từ checkpoint):', predictor.class_names)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Đổi thành ảnh hoa bất kỳ của bạn; samples/ do scripts/make_sample_images.py tạo
img_path = os.path.join(config.SAMPLE_DIR, 'rose.jpg')

result = predictor.predict_image(img_path)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(img_path))
plt.axis('off')
plt.title(f"{result['prediction_vi']} — {result['confidence_percentage']}%")
plt.show()

for info in sorted(result['probabilities'].values(), key=lambda v: -v['percentage']):
    print(f"{info['name_vi']:22s} {info['percentage']:6.2f}%")

## 8. Chạy toàn bộ pipeline bằng command line

Notebook chỉ để minh họa. Quy trình chính thức:

```bash
python download_data.py                 # tải & lọc dữ liệu
python -m src.pipelines.trainer         # huấn luyện
python -m src.pipelines.evaluator       # đánh giá
python -m uvicorn app.main:app --reload # web app
pytest -q                               # kiểm thử
```